In [2]:
%pip install torch torchvision pandas pillow

  Using cached filelock-3.32.5-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.7.0-py3-none-any.whl.metadata (10 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   - -------------------------------------- 3.1/124.1 MB 16.8 MB/s eta 0:00:08
   -- ------------------------------------- 7.3/124.1 MB 18.1 MB/s eta 0:00:07
   --- ------------------------------------ 11.8/124.1 MB 18.9 MB/s eta 0:00:06
   ----- ---------------------------------- 16.5/124.1 MB 20.4 MB/s eta 0:00:06
   ------ --------------------------------- 21.5/124.1 MB 20.9 MB/s eta 0:00:05
   -------- ------------------------------- 26.5/124.1 MB 21.2 MB/s eta 0:00:05
   ---------- ----------------------------- 32.0/124.1 MB 22.1 MB/s eta 0:00:05
   ------------ --------------------------- 38.5/1

In [15]:
import pandas as pd
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class CycloneDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_path = self.data_frame.iloc[idx]['file_path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.data_frame.iloc[idx]['imd_class'])

        if self.transform:
            image = self.transform(image)

        return image, label

In [16]:
from pathlib import Path

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

csv_path = Path.cwd() / "cyclone_dataset" / "processed_cyclone_metadata.csv"
dataset = CycloneDataset(csv_path, transform=data_transforms)
print(f"Dataset samples: {len(dataset)}")
image, label = dataset[0]
print(f"Image shape: {tuple(image.shape)}")
print(f"First label: {label}")

Dataset samples: 136
Image shape: (3, 224, 224)
First label: 0


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models, transforms
from dataset_loader import CycloneDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [10]:

BATCH_SIZE = 16
NUM_CLASSES = 5 
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = CycloneDataset('./cyclone_dataset/processed_cyclone_metadata.csv', transform=data_transforms)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total Images: {len(dataset)} | Training: {train_size} | Validation: {val_size}")

Total Images: 136 | Training: 108 | Validation: 28


In [13]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model successfully initialized and moved to device!")

Model successfully initialized and moved to device!


In [14]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / train_size
    epoch_acc = correct_train / total_train
    model.eval()
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_acc = correct_val / total_val if total_val > 0 else 0.0

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Acc: {val_acc:.4f}")

torch.save(model.state_dict(), './cyclone_ai_model.pth')
print("\nTraining complete! Model saved successfully as './cyclone_ai_model.pth'")

Epoch [1/5] | Loss: 1.8133 | Train Acc: 0.3611 | Val Acc: 0.4286
Epoch [2/5] | Loss: 0.9229 | Train Acc: 0.6852 | Val Acc: 0.3929
Epoch [3/5] | Loss: 0.3318 | Train Acc: 0.8704 | Val Acc: 0.3571
Epoch [4/5] | Loss: 0.1981 | Train Acc: 0.9167 | Val Acc: 0.3214
Epoch [5/5] | Loss: 0.1277 | Train Acc: 0.9444 | Val Acc: 0.2857

Training complete! Model saved successfully as './cyclone_ai_model.pth'
